# 03 — FULL UCF101 tracked ViTPose extraction (GPU-improved, resumable)

This version resumes the existing full extraction output directory and runs YOLO + ViTPose on `cuda:0` **only after a separate GPU preflight succeeds**.

## Critical resume behavior
- Uses the exact same per-video NPZ output folder as the CPU run: `data/tracked_keypoints_iou_v3_full_v1`.
- Previously completed and verified full-video NPZ files are skipped, including those produced on CPU.
- No model training occurs in this notebook. YOLO and ViTPose weights are used for inference only; they are not retrained.
- A video that was interrupted before its final NPZ was atomically saved may need to be processed again from its beginning. Other completed videos are retained.

## GPU settings
- Explicit device: `cuda:0` for both YOLO and ViTPose.
- A one-video / 12-frame GPU smoke test runs first in a separate process. Full extraction will not start unless YOLO **and** ViTPose successfully run on the GPU.
- If CUDA is unavailable or GPU inference fails, stop here. Do not silently fall back to CPU.
- Keeps the original frame step, confidence, image size, max people, IoU tracking method, and source split assignments unchanged.

## Outputs and protection
- Existing full-v1 NPZ outputs are reused in place so resume works.
- New GPU logs and reports are written to separate `_gpu_improved_v1` folders.
- Pilot outputs, Notebook 1 data, model checkpoints, and the teacher's Anaconda installation are not modified.

**Important:** GPU inference may produce tiny floating-point differences from CPU inference, but it uses the same model weights and preprocessing. Each new NPZ records its inference device. Existing CPU files are still accepted if their schema and completeness checks pass.

In [1]:
from pathlib import Path
from datetime import datetime
import json, os, sys, subprocess, hashlib, re, time, gc
import pandas as pd
import torch

# ---------------------------------------------------------------------
# PROJECT PATHS — do not change the existing Notebook 1 environment.
# ---------------------------------------------------------------------
PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
NOTEBOOK2_DIR = PROJECT_DIR / "Notebook_2_Improved_v2"
SPLIT_PATH = PROJECT_DIR / "data" / "processed" / "sequences_group_split_v1" / "video_split_manifest.csv"
VITPOSE_CHECKPOINT = PROJECT_DIR / "checkpoints" / "epoch_209.pth"
YOLO_CANDIDATES = [PROJECT_DIR / "notebooks" / "yolo11n.pt", PROJECT_DIR / "yolo11n.pt"]
YOLO_WEIGHTS = next((p for p in YOLO_CANDIDATES if p.is_file()), None)

# NEW output locations: these do not overwrite the five-video pilot.
OUT_DIR = NOTEBOOK2_DIR / "data" / "tracked_keypoints_iou_v3_full_v1"
# Keep the same NPZ directory so verified CPU-completed videos are reused.
REPORT_DIR = NOTEBOOK2_DIR / "outputs" / "reports" / "tracked_pose_extraction_iou_v3_full_gpu_improved_v1"
LOG_DIR = NOTEBOOK2_DIR / "logs" / "tracked_pose_extraction_iou_v3_full_gpu_improved_v1"
RESULT_DIR = REPORT_DIR / "per_video_results"
SRC_DIR = NOTEBOOK2_DIR / "src"
for folder in [OUT_DIR, REPORT_DIR, LOG_DIR, RESULT_DIR, SRC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

assert SPLIT_PATH.is_file(), f"Frozen group split manifest not found: {SPLIT_PATH}"
assert VITPOSE_CHECKPOINT.is_file(), f"ViTPose checkpoint not found: {VITPOSE_CHECKPOINT}"
assert YOLO_WEIGHTS is not None, "YOLO weights not found in notebooks/yolo11n.pt or project root/yolo11n.pt"

# ---------------------------------------------------------------------
# FULL DATASET CONFIGURATION — this file is intentionally full-run.
# ---------------------------------------------------------------------
RUN_FULL_MANIFEST = True
MAX_DECODED_FRAMES = 0       # 0 = read each complete video to EOF
FRAME_STEP = 4               # ViTPose every 4th decoded frame
MAX_PEOPLE_PER_FRAME = 2     # retain the two largest detections per sampled frame
CONF = 0.20
IMGSZ = 320
MAX_DET = 10
WORKER_DEVICE = "cuda:0"     # explicit NVIDIA GPU for YOLO and ViTPose; preflight required
BATCH_SIZE = 25              # internal batch; no manual 25-video reruns required
WORKER_TIMEOUT_SECONDS = 12 * 60 * 60  # informational; worker is monitored until exit
MAX_TRACK_AGE = 12           # decoded frames
IOU_THRESHOLD = 0.25

# Do not begin if the existing Python environment cannot see the NVIDIA GPU.
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available in this current Jupyter kernel. "
        "The GPU extraction has NOT started and no existing NPZ files were changed. "
        "Check the NVIDIA driver / PyTorch CUDA visibility before continuing."
    )
GPU_NAME = torch.cuda.get_device_name(0)
GPU_FREE_BYTES, GPU_TOTAL_BYTES = torch.cuda.mem_get_info(0)
print("CUDA available: True")
print("GPU:", GPU_NAME)
print("GPU memory free / total (GiB):", round(GPU_FREE_BYTES / 2**30, 2), "/", round(GPU_TOTAL_BYTES / 2**30, 2))

split_df = pd.read_csv(SPLIT_PATH)
required_columns = {"video_path", "label", "split", "video_id", "recording_group_key"}
missing_columns = required_columns.difference(split_df.columns)
assert not missing_columns, f"Manifest is missing columns: {sorted(missing_columns)}"
assert split_df["video_path"].notna().all(), "Manifest contains an empty video_path"
assert split_df["recording_group_key"].notna().all(), "Manifest contains an empty recording_group_key"

# Confirm that the frozen group split itself has no cross-split group overlap.
group_sets = {
    split: set(split_df.loc[split_df["split"].astype(str).str.casefold().eq(split), "recording_group_key"].astype(str))
    for split in ["train", "val", "test"]
}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    overlap = group_sets[a] & group_sets[b]
    assert not overlap, f"Recording-group overlap in source manifest ({a}/{b}): {list(overlap)[:5]}"

# Normalize and de-duplicate source paths; do not change labels or split assignments.
def path_key(value):
    return str(Path(str(value)).resolve()).replace("/", "\\").casefold()

split_df["video_path"] = split_df["video_path"].astype(str)
split_df["label"] = split_df["label"].astype(str)
split_df["split"] = split_df["split"].astype(str).str.casefold()
if split_df["video_path"].map(path_key).duplicated().any():
    duplicates = split_df.loc[split_df["video_path"].map(path_key).duplicated(keep=False), "video_path"].tolist()[:10]
    raise ValueError(f"Duplicate source video paths in manifest: {duplicates}")

if not RUN_FULL_MANIFEST:
    raise RuntimeError("This replacement notebook is configured for the full manifest. Do not silently switch it to pilot mode.")
selected_records = split_df.to_dict("records")
assert len(selected_records) > 0, "The source manifest is empty."

print("FULL UCF101 EXTRACTION")
print("Expected source videos:", len(selected_records))
print("Videos by frozen split:", split_df["split"].value_counts().to_dict())
print("Classes:", split_df["label"].nunique())
print("YOLO weights:", YOLO_WEIGHTS)
print("ViTPose checkpoint:", VITPOSE_CHECKPOINT)
print("Device:", WORKER_DEVICE, "| batch size:", BATCH_SIZE, "| FRAME_STEP:", FRAME_STEP)
print("GPU-improved report directory:", REPORT_DIR)
print("GPU-improved log directory:", LOG_DIR)
print("New tracked NPZ directory:", OUT_DIR)
print("New report directory:", REPORT_DIR)
print("Recording-group overlap: 0")


CUDA available: True
GPU: NVIDIA RTX A4000
GPU memory free / total (GiB): 14.9 / 15.99
FULL UCF101 EXTRACTION
Expected source videos: 13319
Videos by frozen split: {'train': 7999, 'test': 2666, 'val': 2654}
Classes: 101
YOLO weights: C:\Users\admin\ViTPose_Action_Recognition\notebooks\yolo11n.pt
ViTPose checkpoint: C:\Users\admin\ViTPose_Action_Recognition\checkpoints\epoch_209.pth
Device: cuda:0 | batch size: 25 | FRAME_STEP: 4
GPU-improved report directory: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\outputs\reports\tracked_pose_extraction_iou_v3_full_gpu_improved_v1
GPU-improved log directory: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\logs\tracked_pose_extraction_iou_v3_full_gpu_improved_v1
New tracked NPZ directory: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\data\tracked_keypoints_iou_v3_full_v1
New report directory: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\outputs\reports\tracked_pose_extract

In [2]:
WORKER_SOURCE = r'''import os, sys, json, time, gc, traceback, ctypes
from pathlib import Path

# Windows: prefer the OpenMP DLL bundled with this Python's PyTorch before
# importing numerical packages. This is process-local and does not change Anaconda.
torch_lib = Path(sys.prefix) / "Lib" / "site-packages" / "torch" / "lib"
_dll_handles = []
_openmp_handle = None
if os.name == "nt":
    torch_iomp = torch_lib / "libiomp5md.dll"
    if torch_iomp.is_file():
        try:
            _openmp_handle = ctypes.WinDLL(str(torch_iomp))
            print("OPENMP_PRELOADED_FROM", str(torch_iomp), flush=True)
        except OSError as exc:
            print("OPENMP_PRELOAD_WARNING", repr(exc), flush=True)
if torch_lib.is_dir():
    os.environ["PATH"] = str(torch_lib) + os.pathsep + os.environ.get("PATH", "")
    if hasattr(os, "add_dll_directory"):
        try:
            _dll_handles.append(os.add_dll_directory(str(torch_lib)))
        except OSError:
            pass
for name in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[name] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

import numpy as np
import torch
import cv2
from ultralytics import YOLO

cfg = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
cv2.setNumThreads(1)

project = Path(cfg["project_dir"])
sys.path.insert(0, str(project))
from src.vitpose_model import ViTPose, get_affine_transform, decode_heatmaps, IMG_H, IMG_W, IMG_MEAN, IMG_STD

device = torch.device(cfg.get("device", "cuda:0"))
if device.type != "cuda":
    raise RuntimeError(f"GPU-improved worker requires CUDA, but config device is {device}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available inside the worker process")
torch.cuda.set_device(device)
props = torch.cuda.get_device_properties(device)
free_bytes, total_bytes = torch.cuda.mem_get_info(device)
print("GPU_DEVICE", torch.cuda.get_device_name(device), flush=True)
print("GPU_MEMORY_GIB", round(free_bytes / 2**30, 2), "/", round(total_bytes / 2**30, 2), flush=True)
print("BATCH_WORKER_START", "videos", len(cfg["videos"]), "device", device, flush=True)
print("YOLO_WEIGHTS", cfg["yolo_weights"], flush=True)
print("VITPOSE_CHECKPOINT", cfg["vitpose_checkpoint"], flush=True)

# Load each network once per batch, rather than once per video.
yolo = YOLO(cfg["yolo_weights"])
pose = ViTPose().to(device)
try:
    checkpoint = torch.load(cfg["vitpose_checkpoint"], map_location=device, weights_only=True)
except TypeError:
    checkpoint = torch.load(cfg["vitpose_checkpoint"], map_location=device)
if "model_state" not in checkpoint:
    raise KeyError("ViTPose checkpoint has no 'model_state' key")
pose.load_state_dict(checkpoint["model_state"], strict=True)
pose.eval()
print("BATCH_MODELS_LOADED", flush=True)


def box_iou(a, b):
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, float(a[2] - a[0])) * max(0.0, float(a[3] - a[1]))
    area_b = max(0.0, float(b[2] - b[0])) * max(0.0, float(b[3] - b[1]))
    return inter / max(area_a + area_b - inter, 1e-8)


def estimate_pose(rgb, box):
    x1, y1, x2, y2 = map(float, box)
    height, width = rgb.shape[:2]
    x1 = float(np.clip(x1, 0, width - 1))
    x2 = float(np.clip(x2, 0, width))
    y1 = float(np.clip(y1, 0, height - 1))
    y2 = float(np.clip(y2, 0, height))
    bw, bh = x2 - x1, y2 - y1
    if bw < 2 or bh < 2:
        return None
    center = np.asarray([(x1 + x2) / 2, (y1 + y2) / 2], dtype=np.float32)
    scale = max(bw, bh * IMG_W / IMG_H) * 1.25
    trans = get_affine_transform(center, scale, 0.0, (IMG_W, IMG_H))
    crop = cv2.warpAffine(rgb, trans, (IMG_W, IMG_H)).astype(np.float32) / 255.0
    crop = (crop - IMG_MEAN) / IMG_STD
    tensor = torch.from_numpy(crop.transpose(2, 0, 1).copy()).float().unsqueeze(0).to(device)
    with torch.inference_mode():
        heatmaps = pose(tensor)[0].float().cpu().numpy()
    keypoints = decode_heatmaps(heatmaps, center, scale).astype(np.float32)
    del tensor, heatmaps, crop
    return keypoints if keypoints.shape == (17, 3) else None


def atomic_npz_save(path, arrays, retries=6):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + f".{os.getpid()}.tmp.npz")
    np.savez_compressed(tmp, **arrays)
    last_error = None
    for attempt in range(retries):
        try:
            os.replace(str(tmp), str(path))
            return
        except PermissionError as exc:
            last_error = exc
            time.sleep(min(attempt + 1, 5))
    try:
        tmp.unlink(missing_ok=True)
    except OSError:
        pass
    raise last_error


def write_json_atomic(path, obj):
    path = Path(path)
    tmp = path.with_name(path.stem + f".{os.getpid()}.tmp.json")
    tmp.write_text(json.dumps(obj, indent=2), encoding="utf-8")
    os.replace(str(tmp), str(path))


def process_video(rec):
    video_path = Path(str(rec["video_path"]))
    label = str(rec["label"])
    out_path = Path(str(rec["output_path"]))
    result_path = Path(str(rec["result_path"]))
    started = time.time()
    cap = None
    try:
        if not video_path.is_file():
            raise FileNotFoundError(f"Video does not exist: {video_path}")
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise RuntimeError("OpenCV could not open source video")
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)

        frame_step = int(cfg["frame_step"])
        max_decoded = int(cfg["max_decoded_frames"])
        max_people = int(cfg["max_people_per_frame"])
        iou_threshold = float(cfg["iou_threshold"])
        max_track_age = int(cfg["max_track_age"])
        frame_index = 0
        reached_eof = False
        active_tracks = {}
        next_track_id = 1
        track_pose_counts = {}
        track_area_sums = {}
        sampled_frame_indices = []
        sampled_observations = []

        def assign_ids(detections, current_frame):
            nonlocal next_track_id
            stale_ids = [tid for tid, state in active_tracks.items()
                         if current_frame - state["last_frame"] > max_track_age]
            for tid in stale_ids:
                active_tracks.pop(tid, None)
            ids = np.full(len(detections), -1, dtype=np.int64)
            candidates = []
            for det_i, det_box in enumerate(detections):
                for track_id, state in active_tracks.items():
                    overlap = box_iou(det_box, state["box"])
                    if overlap >= iou_threshold:
                        candidates.append((overlap, det_i, track_id))
            candidates.sort(key=lambda item: item[0], reverse=True)
            used_detections, used_tracks = set(), set()
            for overlap, det_i, track_id in candidates:
                if det_i in used_detections or track_id in used_tracks:
                    continue
                ids[det_i] = int(track_id)
                used_detections.add(det_i)
                used_tracks.add(track_id)
            for det_i in range(len(detections)):
                if ids[det_i] < 0:
                    ids[det_i] = next_track_id
                    next_track_id += 1
            for det_i, det_box in enumerate(detections):
                track_id = int(ids[det_i])
                previous = active_tracks.get(track_id, {})
                active_tracks[track_id] = {
                    "box": np.asarray(det_box, dtype=np.float32).copy(),
                    "last_frame": int(current_frame),
                    "hits": int(previous.get("hits", 0)) + 1,
                }
            return ids

        print("VIDEO_START", label, video_path.name, flush=True)
        while True:
            if max_decoded > 0 and frame_index >= max_decoded:
                break
            ok, frame = cap.read()
            if not ok:
                reached_eof = True
                break

            # YOLO runs on each decoded frame to maintain temporal association.
            prediction = yolo.predict(
                frame, classes=[0], conf=float(cfg["conf"]), imgsz=int(cfg["imgsz"]),
                device=str(device), max_det=int(cfg["max_det"]), verbose=False
            )[0]
            if prediction.boxes is not None and len(prediction.boxes) > 0:
                boxes = prediction.boxes.xyxy.cpu().numpy().astype(np.float32)
                confidences = prediction.boxes.conf.cpu().numpy().astype(np.float32)
                ids = assign_ids(boxes, frame_index)
            else:
                boxes = np.empty((0, 4), dtype=np.float32)
                confidences = np.empty((0,), dtype=np.float32)
                ids = assign_ids(boxes, frame_index)

            if frame_index % frame_step == 0:
                frame_observations = []
                if len(boxes) > 0:
                    areas = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
                    keep_indices = np.argsort(areas)[::-1][:max_people]
                    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    for det_i in keep_indices:
                        track_id = int(ids[det_i])
                        keypoints = estimate_pose(rgb, boxes[det_i])
                        if keypoints is None:
                            continue
                        finite_joints = np.isfinite(keypoints[:, :2]).all(axis=1)
                        pose_ok = int(finite_joints.sum()) >= 8
                        frame_observations.append({
                            "track_id": track_id,
                            "keypoints": keypoints,
                            "box": boxes[det_i].astype(np.float32),
                            "detection_score": float(confidences[det_i]),
                            "area": float(areas[det_i]),
                            "pose_ok": bool(pose_ok),
                        })
                        if pose_ok:
                            track_pose_counts[track_id] = track_pose_counts.get(track_id, 0) + 1
                            track_area_sums[track_id] = track_area_sums.get(track_id, 0.0) + float(areas[det_i])
                    del rgb
                sampled_frame_indices.append(int(frame_index))
                sampled_observations.append(frame_observations)

            frame_index += 1
            del prediction
            if frame_index % 500 == 0:
                print("FRAME_PROGRESS", label, "decoded", frame_index, "sampled", len(sampled_frame_indices), flush=True)
                gc.collect()

        cap.release()
        cap = None

        # Select the most consistently pose-observed person; retain all-person data too.
        target_track_id = max(
            track_pose_counts,
            key=lambda tid: (track_pose_counts[tid], track_area_sums.get(tid, 0.0))
        ) if track_pose_counts else -1
        n_samples = len(sampled_frame_indices)
        keypoints_selected = np.full((n_samples, 17, 3), np.nan, dtype=np.float32)
        boxes_selected = np.full((n_samples, 4), np.nan, dtype=np.float32)
        has_person = np.zeros(n_samples, dtype=bool)
        selected_track_ids = np.full(n_samples, -1, dtype=np.int64)
        detection_scores = np.zeros(n_samples, dtype=np.float32)
        all_keypoints, all_boxes, all_frame_indices, all_track_ids, all_detection_scores = [], [], [], [], []

        for sample_i, observations in enumerate(sampled_observations):
            for obs in observations:
                all_keypoints.append(obs["keypoints"])
                all_boxes.append(obs["box"])
                all_frame_indices.append(sampled_frame_indices[sample_i])
                all_track_ids.append(obs["track_id"])
                all_detection_scores.append(obs["detection_score"])
            selected_obs = next((obs for obs in observations
                                 if obs["track_id"] == target_track_id and target_track_id >= 0 and obs["pose_ok"]), None)
            if selected_obs is not None:
                keypoints_selected[sample_i] = selected_obs["keypoints"]
                boxes_selected[sample_i] = selected_obs["box"]
                has_person[sample_i] = True
                selected_track_ids[sample_i] = target_track_id
                detection_scores[sample_i] = selected_obs["detection_score"]

        all_keypoints = np.asarray(all_keypoints, dtype=np.float32).reshape(-1, 17, 3)
        all_boxes = np.asarray(all_boxes, dtype=np.float32).reshape(-1, 4)
        all_frame_indices = np.asarray(all_frame_indices, dtype=np.int32)
        all_track_ids = np.asarray(all_track_ids, dtype=np.int64)
        all_detection_scores = np.asarray(all_detection_scores, dtype=np.float32)

        arrays = {
            "keypoints": keypoints_selected,
            "boxes": boxes_selected,
            "frame_indices": np.asarray(sampled_frame_indices, dtype=np.int32),
            "has_person": has_person,
            "track_ids": selected_track_ids,
            "detection_scores": detection_scores,
            "all_keypoints": all_keypoints,
            "all_boxes": all_boxes,
            "all_frame_indices": all_frame_indices,
            "all_track_ids": all_track_ids,
            "all_detection_scores": all_detection_scores,
            "label": np.asarray(label),
            "video_path": np.asarray(str(video_path)),
            "target_track_id": np.asarray(target_track_id, dtype=np.int64),
            "fps": np.asarray(fps, dtype=np.float32),
            "total_frames": np.asarray(total_frames, dtype=np.int32),
            "frame_step": np.asarray(frame_step, dtype=np.int32),
            "person_selection": np.asarray("most_pose_observations_by_greedy_iou_track"),
            "complete_video": np.asarray(reached_eof, dtype=np.bool_),
            "max_decoded_frames": np.asarray(max_decoded, dtype=np.int32),
            "decoded_frames": np.asarray(frame_index, dtype=np.int32),
            "max_people_per_frame": np.asarray(max_people, dtype=np.int32),
            "inference_device": np.asarray(str(device)),
        }
        atomic_npz_save(out_path, arrays)
        result_obj = {
            "status": "success" if reached_eof else "partial",
            "label": label,
            "video_path": str(video_path),
            "output_path": str(out_path),
            "decoded_frames": int(frame_index),
            "sampled_frames": int(n_samples),
            "detections_with_pose": int(has_person.sum()),
            "pose_success_rate": float(has_person.mean()) if n_samples else 0.0,
            "tracked_ids": int(len(track_pose_counts)),
            "target_track_id": int(target_track_id),
            "target_track_pose_observations": int(track_pose_counts.get(target_track_id, 0)),
            "all_person_observations": int(len(all_track_ids)),
            "inference_device": str(device),
            "complete_video": bool(reached_eof),
            "seconds": round(time.time() - started, 2),
        }
        write_json_atomic(result_path, result_obj)
        print("VIDEO_DONE", json.dumps(result_obj, ensure_ascii=False), flush=True)
        return result_obj
    except Exception as exc:
        if cap is not None:
            cap.release()
        error_obj = {
            "status": "video_error",
            "label": label,
            "video_path": str(video_path),
            "output_path": str(out_path),
            "complete_video": False,
            "error": repr(exc),
            "traceback": traceback.format_exc(limit=8),
            "seconds": round(time.time() - started, 2),
        }
        write_json_atomic(result_path, error_obj)
        print("VIDEO_ERROR", json.dumps(error_obj, ensure_ascii=False), flush=True)
        return error_obj
    finally:
        if cap is not None:
            cap.release()
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


batch_results = []
for video_record in cfg["videos"]:
    batch_results.append(process_video(video_record))
print("BATCH_DONE", json.dumps({"videos": len(batch_results), "success": sum(r.get("status") == "success" for r in batch_results), "errors": sum(r.get("status") != "success" for r in batch_results)}), flush=True)
'''

# ---------------------------------------------------------------------
# Create the worker script and verify whether an existing NPZ is complete.
# The worker loads YOLO + ViTPose once per 25-video batch.
# ---------------------------------------------------------------------
WORKER_PATH = SRC_DIR / "tracked_vitpose_worker_iou_gpu_improved_v1.py"
WORKER_PATH.write_text(WORKER_SOURCE, encoding="utf-8")


def stable_video_id(video_path):
    return hashlib.sha1(path_key(video_path).encode("utf-8")).hexdigest()[:16]


def output_npz_path(record):
    label_safe = re.sub(r"[^A-Za-z0-9_-]+", "_", str(record["label"])).strip("_") or "unknown"
    digest = stable_video_id(record["video_path"])
    return OUT_DIR / f"{label_safe}__{Path(str(record['video_path'])).stem}__{digest}.npz"


def result_json_path(record):
    return RESULT_DIR / f"{stable_video_id(record['video_path'])}.json"


def existing_npz_is_complete(npz_path, expected_video_path, expected_label):
    if not npz_path.is_file():
        return False
    required = {
        "keypoints", "boxes", "frame_indices", "has_person", "track_ids",
        "all_keypoints", "all_boxes", "all_frame_indices", "all_track_ids",
        "label", "video_path", "complete_video", "decoded_frames",
        "max_decoded_frames", "frame_step", "max_people_per_frame", "target_track_id"
    }
    try:
        with np.load(npz_path, allow_pickle=False) as d:
            if not required.issubset(d.files):
                return False
            n = len(d["frame_indices"])
            valid = (
                d["keypoints"].shape == (n, 17, 3)
                and d["boxes"].shape == (n, 4)
                and d["has_person"].shape == (n,)
                and d["track_ids"].shape == (n,)
                and d["all_keypoints"].ndim == 3
                and d["all_keypoints"].shape[1:] == (17, 3)
                and len(d["all_track_ids"]) == len(d["all_keypoints"])
                and str(d["video_path"].item())
                    .replace("/", "\\").casefold()
                    == str(expected_video_path).replace("/", "\\").casefold()
                and str(d["label"].item()) == str(expected_label)
                and bool(d["complete_video"].item())
                and int(d["max_decoded_frames"].item()) == 0
                and int(d["frame_step"].item()) == FRAME_STEP
                and int(d["max_people_per_frame"].item()) == MAX_PEOPLE_PER_FRAME
                and int(d["decoded_frames"].item()) > 0
            )
            return bool(valid)
    except Exception:
        return False


print("Worker script:", WORKER_PATH)
print("Full output directory:", OUT_DIR)
print("The previous five-video pilot directory is not used or overwritten.")


Worker script: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\src\tracked_vitpose_worker_iou_gpu_improved_v1.py
Full output directory: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\data\tracked_keypoints_iou_v3_full_v1
The previous five-video pilot directory is not used or overwritten.


In [3]:
# ---------------------------------------------------------------------
# GPU SMOKE TEST — one known Typing video, 12 decoded frames only.
# This verifies YOLO + ViTPose both execute on CUDA before the full queue starts.
# It writes only inside the GPU diagnostic log folder, never into the main NPZ folder.
# ---------------------------------------------------------------------
GPU_PREFLIGHT_DIR = LOG_DIR / "gpu_preflight"
GPU_PREFLIGHT_DIR.mkdir(parents=True, exist_ok=True)

# Prefer a known class/video used successfully in the earlier pilot.
typing_rows = split_df[
    split_df["label"].astype(str).str.casefold().eq("typing")
    & split_df["video_path"].map(lambda p: Path(str(p)).is_file())
]
if len(typing_rows):
    preflight_row = typing_rows.iloc[0]
else:
    valid_rows = split_df[split_df["video_path"].map(lambda p: Path(str(p)).is_file())]
    if valid_rows.empty:
        raise FileNotFoundError("No existing source video is available for the GPU smoke test.")
    preflight_row = valid_rows.iloc[0]

smoke_cfg_path = GPU_PREFLIGHT_DIR / "gpu_smoke_config.json"
smoke_stdout_path = GPU_PREFLIGHT_DIR / "gpu_smoke_stdout.log"
smoke_stderr_path = GPU_PREFLIGHT_DIR / "gpu_smoke_stderr.log"
smoke_npz_path = GPU_PREFLIGHT_DIR / "gpu_smoke_test.npz"
smoke_json_path = GPU_PREFLIGHT_DIR / "gpu_smoke_test.json"

smoke_config = {
    "project_dir": str(PROJECT_DIR),
    "yolo_weights": str(YOLO_WEIGHTS),
    "vitpose_checkpoint": str(VITPOSE_CHECKPOINT),
    "device": WORKER_DEVICE,
    "frame_step": FRAME_STEP,
    "max_decoded_frames": 12,
    "max_people_per_frame": MAX_PEOPLE_PER_FRAME,
    "conf": CONF,
    "imgsz": IMGSZ,
    "max_det": MAX_DET,
    "iou_threshold": IOU_THRESHOLD,
    "max_track_age": MAX_TRACK_AGE,
    "videos": [{
        "video_path": str(preflight_row["video_path"]),
        "label": str(preflight_row["label"]),
        "output_path": str(smoke_npz_path),
        "result_path": str(smoke_json_path),
    }],
}
smoke_cfg_path.write_text(json.dumps(smoke_config, indent=2), encoding="utf-8")
smoke_env = os.environ.copy()
for key in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    smoke_env[key] = "1"
torch_lib = Path(sys.prefix) / "Lib" / "site-packages" / "torch" / "lib"
if torch_lib.is_dir():
    smoke_env["PATH"] = str(torch_lib) + os.pathsep + smoke_env.get("PATH", "")

print("Starting one-video CUDA smoke test. Full extraction has NOT started yet.")
with smoke_stdout_path.open("w", encoding="utf-8", errors="replace") as out_f, \
     smoke_stderr_path.open("w", encoding="utf-8", errors="replace") as err_f:
    smoke_proc = subprocess.run(
        [sys.executable, str(WORKER_PATH), str(smoke_cfg_path)],
        cwd=str(PROJECT_DIR), env=smoke_env,
        stdout=out_f, stderr=err_f,
        timeout=15 * 60,
    )

print("GPU smoke-test return code:", smoke_proc.returncode)
print("STDOUT log:", smoke_stdout_path)
print("STDERR log:", smoke_stderr_path)
print("--- worker stdout ---")
print(smoke_stdout_path.read_text(encoding="utf-8", errors="replace")[-6000:])
print("--- worker stderr ---")
print(smoke_stderr_path.read_text(encoding="utf-8", errors="replace")[-4000:])

if smoke_proc.returncode != 0:
    raise RuntimeError(
        "GPU smoke test failed. Full extraction was NOT started. "
        f"Inspect {smoke_stderr_path} and {smoke_stdout_path}."
    )
if not smoke_json_path.is_file():
    raise RuntimeError("GPU smoke test produced no result JSON. Full extraction was NOT started.")
smoke_result = json.loads(smoke_json_path.read_text(encoding="utf-8"))
if smoke_result.get("status") != "partial":
    raise RuntimeError(f"GPU smoke test returned an unexpected status: {smoke_result.get('status')}")
if smoke_result.get("inference_device") != WORKER_DEVICE:
    raise RuntimeError("Result JSON does not confirm the requested CUDA device.")
if int(smoke_result.get("detections_with_pose", 0)) <= 0:
    raise RuntimeError(
        "YOLO/ViTPose did not produce a valid pose in the GPU smoke test. "
        "Full extraction was NOT started. Check the chosen sample and worker logs."
    )
GPU_PREFLIGHT_OK = True
print("\nGPU PREFLIGHT PASSED")
print("GPU:", GPU_NAME)
print("Test video:", smoke_result.get("video_path"))
print("Decoded frames:", smoke_result.get("decoded_frames"))
print("Sampled frames:", smoke_result.get("sampled_frames"))
print("Pose successes:", smoke_result.get("detections_with_pose"))
print("Inference device:", smoke_result.get("inference_device"))
print("The full extraction may now be started in the next cell.")

Starting one-video CUDA smoke test. Full extraction has NOT started yet.
GPU smoke-test return code: 0
STDOUT log: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\logs\tracked_pose_extraction_iou_v3_full_gpu_improved_v1\gpu_preflight\gpu_smoke_stdout.log
STDERR log: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\logs\tracked_pose_extraction_iou_v3_full_gpu_improved_v1\gpu_preflight\gpu_smoke_stderr.log
--- worker stdout ---
OPENMP_PRELOADED_FROM C:\Users\admin\anaconda3\Lib\site-packages\torch\lib\libiomp5md.dll
GPU_DEVICE NVIDIA RTX A4000
GPU_MEMORY_GIB 14.9 / 15.99
BATCH_WORKER_START videos 1 device cuda:0
YOLO_WEIGHTS C:\Users\admin\ViTPose_Action_Recognition\notebooks\yolo11n.pt
VITPOSE_CHECKPOINT C:\Users\admin\ViTPose_Action_Recognition\checkpoints\epoch_209.pth
BATCH_MODELS_LOADED
VIDEO_START Typing v_Typing_g01_c01.avi
VIDEO_DONE {"status": "partial", "label": "Typing", "video_path": "C:\\Users\\admin\\ViTPose_Action_Recognition\\data\\raw\\uc

In [ ]:
# ---------------------------------------------------------------------
# FULL RESUMABLE GPU EXTRACTION
# - All 13,319 manifest videos are selected automatically.
# - Completed NPZ files are verified and skipped.
# - Each worker process handles up to BATCH_SIZE videos, loading models once.
# - Every completed video's NPZ + JSON is saved immediately.
# - A native worker crash stops the run safely; rerun to resume.
# ---------------------------------------------------------------------

assert GPU_PREFLIGHT_OK, "GPU preflight has not passed. Do not start the full extraction."

PROGRESS_CSV = REPORT_DIR / "tracked_pose_extraction_full_gpu_progress.csv"
BATCH_REPORT_CSV = REPORT_DIR / "tracked_pose_extraction_full_gpu_report.csv"

# Build queue of videos that do not yet have a verified complete full-video NPZ.
pending_records = []
skipped_rows = []
for i, rec in enumerate(selected_records, 1):
    npz_path = output_npz_path(rec)
    if existing_npz_is_complete(npz_path, rec["video_path"], rec["label"]):
        skipped_rows.append({
            "status": "skipped_existing_verified",
            "label": rec["label"],
            "split": rec["split"],
            "video_id": rec["video_id"],
            "recording_group_key": rec["recording_group_key"],
            "video_path": rec["video_path"],
            "output_path": str(npz_path),
            "complete_video": True,
        })
    else:
        pending_records.append(rec)
    if i % 1000 == 0:
        print(f"Queue checked: {i}/{len(selected_records)} | pending={len(pending_records)} | verified existing={len(skipped_rows)}")

print("\nManifest videos:", len(selected_records))
print("Already complete and verified:", len(skipped_rows))
print("Videos remaining:", len(pending_records))

# Persist skipped records too, so the final CSV represents the complete manifest.
progress_rows = list(skipped_rows)
for row in skipped_rows:
    rec = next(r for r in selected_records if path_key(r["video_path"]) == path_key(row["video_path"]))
    result_json_path(rec).write_text(json.dumps(row, indent=2), encoding="utf-8")

batch_count = (len(pending_records) + BATCH_SIZE - 1) // BATCH_SIZE
print("Worker batches remaining:", batch_count)
print("Starting full extraction. Keep the computer plugged in and awake.")

for batch_index, start in enumerate(range(0, len(pending_records), BATCH_SIZE), 1):
    batch_records = pending_records[start:start + BATCH_SIZE]
    worker_video_records = []
    for rec in batch_records:
        worker_video_records.append({
            "video_path": str(rec["video_path"]),
            "label": str(rec["label"]),
            "output_path": str(output_npz_path(rec)),
            "result_path": str(result_json_path(rec)),
        })

    batch_id = f"batch_{batch_index:04d}"
    config_path = LOG_DIR / f"{batch_id}_config.json"
    stdout_path = LOG_DIR / f"{batch_id}_stdout.log"
    stderr_path = LOG_DIR / f"{batch_id}_stderr.log"
    config = {
        "project_dir": str(PROJECT_DIR),
        "yolo_weights": str(YOLO_WEIGHTS),
        "vitpose_checkpoint": str(VITPOSE_CHECKPOINT),
        "device": WORKER_DEVICE,
        "frame_step": FRAME_STEP,
        "max_decoded_frames": MAX_DECODED_FRAMES,
        "max_people_per_frame": MAX_PEOPLE_PER_FRAME,
        "conf": CONF,
        "imgsz": IMGSZ,
        "max_det": MAX_DET,
        "iou_threshold": IOU_THRESHOLD,
        "max_track_age": MAX_TRACK_AGE,
        "videos": worker_video_records,
    }
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

    print(f"\n=== BATCH {batch_index}/{batch_count}: {len(batch_records)} videos ===", flush=True)
    env = os.environ.copy()
    for key in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
        env[key] = "1"
    torch_lib = Path(sys.prefix) / "Lib" / "site-packages" / "torch" / "lib"
    if torch_lib.is_dir():
        env["PATH"] = str(torch_lib) + os.pathsep + env.get("PATH", "")

    return_code = None
    proc = None
    try:
        with stdout_path.open("w", encoding="utf-8", errors="replace") as stdout_file, \
             stderr_path.open("w", encoding="utf-8", errors="replace") as stderr_file:
            proc = subprocess.Popen(
                [sys.executable, str(WORKER_PATH), str(config_path)],
                cwd=str(PROJECT_DIR), env=env,
                stdout=subprocess.PIPE, stderr=stderr_file,
                text=True, encoding="utf-8", errors="replace", bufsize=1
            )
            try:
                for line in proc.stdout:
                    stdout_file.write(line)
                    stdout_file.flush()
                    stripped = line.strip()
                    if stripped.startswith(("GPU_DEVICE", "GPU_MEMORY_GIB", "BATCH_WORKER_START", "BATCH_MODELS_LOADED", "VIDEO_START", "VIDEO_DONE", "VIDEO_ERROR", "FRAME_PROGRESS", "BATCH_DONE")):
                        print(stripped[:1200], flush=True)
            except KeyboardInterrupt:
                print("\nSTOP REQUESTED: terminating only the active GPU worker...", flush=True)
                if proc.poll() is None:
                    proc.terminate()
                    try:
                        proc.wait(timeout=15)
                    except subprocess.TimeoutExpired:
                        proc.kill()
                        proc.wait(timeout=10)
                print("Active worker stopped. Completed per-video NPZ files remain saved.", flush=True)
                print("The video interrupted before its final NPZ save will be retried on resume.", flush=True)
                raise
            return_code = proc.wait()
    except KeyboardInterrupt:
        print("\nExtraction safely interrupted by user. Do not delete output folders.", flush=True)
        raise
    finally:
        if proc is not None and proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait(timeout=10)

    # Rebuild progress from persistent per-video JSON files after every batch.
    result_rows = []
    for result_file in RESULT_DIR.glob("*.json"):
        try:
            result_rows.append(json.loads(result_file.read_text(encoding="utf-8")))
        except Exception as exc:
            print("Warning: unreadable per-video result JSON:", result_file.name, repr(exc))
    report_df = pd.DataFrame(result_rows)
    if not report_df.empty:
        # Add manifest split and group metadata without modifying the frozen manifest.
        meta_lookup = {path_key(r["video_path"]): r for r in selected_records}
        report_df["split"] = report_df["video_path"].map(lambda p: meta_lookup.get(path_key(p), {}).get("split", "unknown"))
        report_df["video_id"] = report_df["video_path"].map(lambda p: meta_lookup.get(path_key(p), {}).get("video_id", -1))
        report_df["recording_group_key"] = report_df["video_path"].map(lambda p: meta_lookup.get(path_key(p), {}).get("recording_group_key", "unknown"))
        report_df.to_csv(PROGRESS_CSV, index=False)
        report_df.to_csv(BATCH_REPORT_CSV, index=False)

    batch_success = sum(1 for r in result_rows if r.get("status") == "success")
    batch_errors = sum(1 for r in result_rows if r.get("status") not in {"success", "skipped_existing_verified"})
    print(f"Batch {batch_index} worker return code: {return_code}")
    print(f"Recorded successful videos so far: {batch_success} | recorded non-success results: {batch_errors}")
    print("Worker stdout:", stdout_path)
    print("Worker stderr:", stderr_path)

    if return_code != 0:
        print("\nWORKER PROCESS FAILED. Completed video NPZ/JSON files are preserved.")
        print("Do not delete the output folder. Fix the worker issue, then rerun this notebook to resume.")
        raise RuntimeError(f"Worker batch {batch_id} exited with code {return_code}. See {stderr_path}")

    # If the batch process exited normally, continue to the next batch.
    del proc
    gc.collect()

print("\nALL MANIFEST BATCHES HAVE BEEN ATTEMPTED.")
print("Progress CSV:", PROGRESS_CSV)

Queue checked: 1000/13319 | pending=1000 | verified existing=0
Queue checked: 2000/13319 | pending=2000 | verified existing=0
Queue checked: 3000/13319 | pending=3000 | verified existing=0
Queue checked: 4000/13319 | pending=4000 | verified existing=0
Queue checked: 5000/13319 | pending=5000 | verified existing=0
Queue checked: 6000/13319 | pending=6000 | verified existing=0
Queue checked: 7000/13319 | pending=7000 | verified existing=0
Queue checked: 8000/13319 | pending=8000 | verified existing=0
Queue checked: 9000/13319 | pending=9000 | verified existing=0
Queue checked: 10000/13319 | pending=10000 | verified existing=0
Queue checked: 11000/13319 | pending=11000 | verified existing=0
Queue checked: 12000/13319 | pending=12000 | verified existing=0
Queue checked: 13000/13319 | pending=13000 | verified existing=0

Manifest videos: 13319
Already complete and verified: 0
Videos remaining: 13319
Worker batches remaining: 533
Starting full extraction. Keep the computer plugged in and awa

In [ ]:
# ---------------------------------------------------------------------
# FINAL COVERAGE AUDIT — distinguish completed videos from errors/missing.
# ---------------------------------------------------------------------
result_rows = []
for result_file in RESULT_DIR.glob("*.json"):
    try:
        result_rows.append(json.loads(result_file.read_text(encoding="utf-8")))
    except Exception as exc:
        print("Unreadable result:", result_file, repr(exc))

results_df = pd.DataFrame(result_rows)
if not results_df.empty:
    meta_lookup = {path_key(r["video_path"]): r for r in selected_records}
    results_df["split"] = results_df["video_path"].map(lambda p: meta_lookup.get(path_key(p), {}).get("split", "unknown"))
    results_df["video_id"] = results_df["video_path"].map(lambda p: meta_lookup.get(path_key(p), {}).get("video_id", -1))
    results_df["recording_group_key"] = results_df["video_path"].map(lambda p: meta_lookup.get(path_key(p), {}).get("recording_group_key", "unknown"))
    results_df.to_csv(PROGRESS_CSV, index=False)

# Independently inspect the output NPZs to count only verified complete files.
complete_count = 0
for rec in selected_records:
    if existing_npz_is_complete(output_npz_path(rec), rec["video_path"], rec["label"]):
        complete_count += 1

status_counts = results_df["status"].value_counts().to_dict() if not results_df.empty else {}
metadata = {
    "experiment": "FULL UCF101 GPU tracked pose extraction: YOLO predict + greedy IoU association + ViTPose",
    "source_group_split_manifest": str(SPLIT_PATH),
    "tracked_keypoint_directory": str(OUT_DIR),
    "report_directory": str(REPORT_DIR),
    "expected_source_videos": int(len(selected_records)),
    "verified_complete_source_videos": int(complete_count),
    "coverage": float(complete_count / max(1, len(selected_records))),
    "is_full_dataset": bool(complete_count == len(selected_records)),
    "status_counts": status_counts,
    "frame_step": int(FRAME_STEP),
    "max_decoded_frames": int(MAX_DECODED_FRAMES),
    "max_people_per_frame": int(MAX_PEOPLE_PER_FRAME),
    "device": WORKER_DEVICE,
    "batch_size": int(BATCH_SIZE),
    "recording_group_split_reused_without_changes": True,
    "tracking_method": "greedy IoU association on YOLO predict detections; not ByteTrack",
    "person_selection": "most pose-valid observations by track; track ID is metadata, not a model feature",
    "warning": "Only use as full tracked dataset if verified_complete_source_videos equals expected_source_videos and coverage equals 1.0. Review class support and pose quality before training."
}
metadata_path = REPORT_DIR / "full_extraction_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("FULL EXTRACTION COVERAGE AUDIT")
print(json.dumps(metadata, indent=2))
print("\nReport:", PROGRESS_CSV)
print("Metadata:", metadata_path)

if complete_count != len(selected_records):
    print("\nNOT COMPLETE YET: rerun this notebook to resume. Verified complete NPZ files will be skipped.")
else:
    print("\nALL MANIFEST VIDEOS HAVE VERIFIED COMPLETE NPZ OUTPUTS.")


## After GPU extraction

1. Check `full_extraction_metadata.json`. A complete run requires `verified_complete_source_videos == expected_source_videos` and `coverage == 1.0`.
2. Review per-video pose-success rates and errors.
3. Previously completed CPU NPZ files in the shared `_full_v1` directory are reused and skipped; new files record `inference_device=cuda:0`.
4. If interrupted, rerun this GPU-improved notebook. Verified complete files are skipped. A video interrupted before its final NPZ save is retried from its beginning.
5. Do not run the old pilot sequence builder. Use a matching builder for `tracked_keypoints_iou_v3_full_v1` after full extraction is complete.

The full extraction cell catches a Jupyter interrupt, terminates its active child worker, and preserves all completed per-video files.